In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 동영상 열기
video_path = None
cap = cv2.VideoCapture(video_path) # 비디오 캡처 객체 생성
if not cap.isOpened():
    print("ERR : 에러 발생(비디오 파일을 찾을 수가 없습니다)")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f'fps: {fps}, width: {width},height: {height},frame_count: {frame_count},')

while True:
    result, frame  = cap.read() # 성공 여부, 이미지(프레임)
    if not result: # 마지막 프레임이면
        break        # 종료해
    cv2.imshow('Gray Video', frame)

    if cv2.waitKey(41) & 0xFF == ord('q'): # 41 ms 만큼 기다리기
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# 중심 구하기
mask = None
moments = cv2.moments(mask) # key-value {}로 저장 (m00: ...  면적(area))

m00 = moments['m00'] # 전체 면적

centroid_x = int(moments['m10']/m00) # 한 영역의 중심 x 좌표
centroid_y = int(moments['m01']/m00) # 한 영역의 중심 y 좌표


In [ ]:
# 코너 검출
gray = None
cv2.cornerHarris(np.float32(gray),blockSize=2,ksize=3, k=0.04) # 미분 계산, k는 민감도 return 점수맵 반환

binary_ad = None
cv2.goodFeaturesToTrack( # 점을 뽑아줌
    binary_ad,
    maxCorners=50,
    qualityLevel=0.01,
    minDistance=10
)


In [ ]:
# 허브 변환 쓸만할지도?
circles =\
cv2.HoughCircles(gray,                  # gray
                 cv2.HOUGH_GRADIENT,    # 기울기 사용 원을 찾겠다.
                 dp=1.2,                # 해상도 scaling 비율, 1.0 입력 이미지와 같은 해상도
                 minDist=30,            # 검출된 원(중심) 사이 최소 거리 >> 원 중심 간 간격이 30이상 >> 중복이 아니다
                 param1=100,            # 높은 임계값(canny edge upper threshold)
                 param2=30,             # 투표(voting) 누적된 임계값(원이라고 판단할 기준)
                 minRadius=10,          # 최소 반지름
                 maxRadius=50           # 최대 반지름
                 )
edges = None
lines = cv2.HoughLinesP(
    edges, # 캐니 엣지로 검출된 멧지 이미지
    rho=1, # 해상도
    theta=np.pi/180, # 해상도(각도)
    threshold=100, # 직선으로 간주될 수 있는 최소값
    minLineLength=100, # 내가 검출하려고 하는 직선의 최소 길이
    maxLineGap=10 # 직선으로 간주되는 간격
)


In [ ]:
# 윤곽선(컨투어) 검출
binary = None
cons = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
# binary : 임계값 적용해서 나온 이진화된 이미지
# RETR_LIST : retrieve list 목록을 검색하다. (모든 윤곽선 검색해)
# CHAIN_APPROX_NONE : 모든 점 저장
# 흰색(255) 영역의 경계선 찾아요 >> 리스트 형식으로 변환
con_packs = cons[0] if len(cons) == 2 else cons[1]
# 시각화
image = None
_ = cv2.drawContours(image,con_packs,-1,(0,255,0),2)
# -1 모든 윤곽선 그려줘
plt.imshow(_)